# Hospital Analytics — Python Data Preparation & EDA

**Project workflow:** Raw CSV files → Python data quality checks & cleaning → SQL Server → Power BI

This notebook documents the Python stage of the Hospital Analytics project. It loads the seven relational datasets, profiles their structure, checks data quality, applies the cleaning rules used for the project, exports cleaned CSV files, and performs a small set of exploratory analyses.

> **GitHub note:** The notebook is intentionally output-free so it renders cleanly on GitHub. Run the cells locally to reproduce the analysis.

## 1. Imports & configuration

The notebook uses only commonly available Python data-analysis libraries.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2. Project paths

Expected repository structure:

```text
hospital-analytics/
├── data/
│   ├── admissions.csv
│   ├── appointments.csv
│   ├── beds.csv
│   ├── billing.csv
│   ├── doctors.csv
│   ├── patients.csv
│   └── treatments.csv
└── python/
    └── hospital_analytics.ipynb
```

The path logic works whether the notebook is opened from the `python/` folder or from the repository root.

In [ ]:
REQUIRED_FILES = [
    "patients.csv",
    "doctors.csv",
    "appointments.csv",
    "admissions.csv",
    "treatments.csv",
    "billing.csv",
    "beds.csv",
]

candidate_data_dirs = [
    Path("../data"),
    Path("data"),
]

DATA_DIR = next((p for p in candidate_data_dirs if p.exists()), Path("../data"))
CLEAN_DIR = DATA_DIR / "cleaned"

missing_files = [name for name in REQUIRED_FILES if not (DATA_DIR / name).exists()]

if missing_files:
    raise FileNotFoundError(
        f"Missing files in {DATA_DIR.resolve()}: {missing_files}"
    )

CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Cleaned-data directory: {CLEAN_DIR.resolve()}")

## 3. Load the relational datasets

In [ ]:
patients = pd.read_csv(DATA_DIR / "patients.csv")
doctors = pd.read_csv(DATA_DIR / "doctors.csv")
appointments = pd.read_csv(DATA_DIR / "appointments.csv")
admissions = pd.read_csv(DATA_DIR / "admissions.csv")
treatments = pd.read_csv(DATA_DIR / "treatments.csv")
billing = pd.read_csv(DATA_DIR / "billing.csv")
beds = pd.read_csv(DATA_DIR / "beds.csv")

tables = {
    "patients": patients,
    "doctors": doctors,
    "appointments": appointments,
    "admissions": admissions,
    "treatments": treatments,
    "billing": billing,
    "beds": beds,
}

print("Loaded tables:", ", ".join(tables))

## 4. Dataset overview

In [ ]:
overview = pd.DataFrame(
    {
        "table": list(tables.keys()),
        "rows": [df.shape[0] for df in tables.values()],
        "columns": [df.shape[1] for df in tables.values()],
    }
)

overview

In [ ]:
for name, df in tables.items():
    print(f"\n{name.upper()} — shape: {df.shape}")
    display(df.dtypes.to_frame("data_type"))

## 5. Data quality checks

The checks below cover:

- missing values
- duplicate primary keys
- invalid age and waiting-time values
- invalid financial values
- referential integrity between foreign keys and parent tables
- date sequencing
- billing reconciliation
- bed/admission consistency
- appointment and treatment timing
- department consistency
- insurance/payment consistency

In [ ]:
# Missing values
missing_summary = pd.DataFrame(
    {
        name: df.isna().sum()
        for name, df in tables.items()
    }
).fillna(0).astype(int)

missing_summary

In [ ]:
# Primary-key duplicate checks
primary_keys = {
    "patients": "patient_id",
    "doctors": "doctor_id",
    "appointments": "appointment_id",
    "admissions": "admission_id",
    "treatments": "treatment_id",
    "billing": "bill_id",
    "beds": "bed_id",
}

duplicate_summary = pd.DataFrame(
    [
        {
            "table": name,
            "primary_key": key,
            "duplicate_rows": int(df[key].duplicated().sum()),
        }
        for name, key in primary_keys.items()
        for df in [tables[name]]
    ]
)

duplicate_summary

In [ ]:
# Basic categorical distributions
categorical_checks = {
    "patient_gender": patients["gender"].value_counts(),
    "insurance_type": patients["insurance_type"].value_counts(),
    "appointment_status": appointments["appointment_status"].value_counts(),
    "admission_type": admissions["admission_type"].value_counts(),
    "bed_type": admissions["bed_type"].value_counts(),
    "payment_status": billing["payment_status"].value_counts(),
    "bed_status": beds["status"].value_counts(),
}

categorical_checks

In [ ]:
# Numeric range checks
quality_checks = {
    "patients_age_below_0": int((patients["age"] < 0).sum()),
    "patients_age_above_100": int((patients["age"] > 100).sum()),
    "appointments_wait_below_0": int((appointments["wait_time_minutes"] < 0).sum()),
    "treatments_cost_non_positive": int((treatments["treatment_cost"] <= 0).sum()),
    "billing_total_non_positive": int((billing["total_amount"] <= 0).sum()),
    "billing_insurance_negative": int((billing["insurance_amount"] < 0).sum()),
    "billing_patient_payment_negative": int((billing["patient_payment"] < 0).sum()),
}

pd.Series(quality_checks, name="issue_count")

### 5.1 Referential-integrity checks

In [ ]:
referential_checks = {
    "appointments → patients": int(
        (~appointments["patient_id"].isin(patients["patient_id"])).sum()
    ),
    "appointments → doctors": int(
        (~appointments["doctor_id"].isin(doctors["doctor_id"])).sum()
    ),
    "admissions → patients": int(
        (~admissions["patient_id"].isin(patients["patient_id"])).sum()
    ),
    "admissions → doctors": int(
        (~admissions["doctor_id"].isin(doctors["doctor_id"])).sum()
    ),
    "treatments → admissions": int(
        (~treatments["admission_id"].isin(admissions["admission_id"])).sum()
    ),
    "billing → patients": int(
        (~billing["patient_id"].isin(patients["patient_id"])).sum()
    ),
    "billing → admissions": int(
        (~billing["admission_id"].isin(admissions["admission_id"])).sum()
    ),
    "beds → admissions": int(
        (
            beds["admission_id"].notna()
            & ~beds["admission_id"].isin(admissions["admission_id"])
        ).sum()
    ),
}

pd.Series(referential_checks, name="unmatched_rows")

### 5.2 Date and business-rule checks

In [ ]:
# Convert date columns before chronological validation.
date_columns = {
    "patients": ["registration_date"],
    "appointments": ["appointment_date"],
    "admissions": ["admission_date", "discharge_date"],
    "treatments": ["treatment_date"],
}

for table_name, columns in date_columns.items():
    for column in columns:
        tables[table_name][column] = pd.to_datetime(
            tables[table_name][column],
            dayfirst=True,
            errors="coerce",
        )

patients = tables["patients"]
appointments = tables["appointments"]
admissions = tables["admissions"]
treatments = tables["treatments"]
billing = tables["billing"]
beds = tables["beds"]

In [ ]:
date_quality = {
    "admissions_discharge_before_admission": int(
        (admissions["discharge_date"] < admissions["admission_date"]).sum()
    ),
    "appointments_before_patient_registration": 0,
    "admissions_before_patient_registration": 0,
    "treatments_before_admission": 0,
    "treatments_after_discharge": 0,
}

appointment_check = appointments.merge(
    patients[["patient_id", "registration_date"]],
    on="patient_id",
    how="left",
)

admission_check = admissions.merge(
    patients[["patient_id", "registration_date"]],
    on="patient_id",
    how="left",
)

treatment_check = treatments.merge(
    admissions[["admission_id", "admission_date", "discharge_date"]],
    on="admission_id",
    how="left",
)

date_quality["appointments_before_patient_registration"] = int(
    (
        appointment_check["appointment_date"]
        < appointment_check["registration_date"]
    ).sum()
)

date_quality["admissions_before_patient_registration"] = int(
    (
        admission_check["admission_date"]
        < admission_check["registration_date"]
    ).sum()
)

date_quality["treatments_before_admission"] = int(
    (
        treatment_check["treatment_date"]
        < treatment_check["admission_date"]
    ).sum()
)

date_quality["treatments_after_discharge"] = int(
    (
        treatment_check["treatment_date"]
        > treatment_check["discharge_date"]
    ).sum()
)

pd.Series(date_quality, name="issue_count")

In [ ]:
# Billing reconciliation
billing["calculated_total"] = (
    billing["insurance_amount"] + billing["patient_payment"]
)

billing_reconciliation_issues = int(
    (~np.isclose(
        billing["calculated_total"],
        billing["total_amount"],
        equal_nan=False,
        atol=0.01,
    )).sum()
)

print(f"Billing reconciliation issues: {billing_reconciliation_issues}")

In [ ]:
# Department consistency between appointments and doctors
doctor_department_check = appointments.merge(
    doctors[["doctor_id", "department"]].rename(
        columns={"department": "doctor_department"}
    ),
    on="doctor_id",
    how="left",
)

department_mismatches = doctor_department_check[
    doctor_department_check["department"]
    != doctor_department_check["doctor_department"]
]

print(f"Department mismatches: {len(department_mismatches):,}")

In [ ]:
# Insurance consistency checks
insurance_check = billing.merge(
    patients[["patient_id", "insurance_type"]],
    on="patient_id",
    how="left",
)

self_pay_with_insurance = insurance_check[
    (insurance_check["insurance_type"] == "Self-Pay")
    & (insurance_check["insurance_amount"] > 0)
]

non_self_pay_without_insurance = insurance_check[
    (insurance_check["insurance_type"] != "Self-Pay")
    & (insurance_check["insurance_amount"] == 0)
]

print(f"Self-Pay records with insurance amount > 0: {len(self_pay_with_insurance):,}")
print(
    "Non-Self-Pay records with zero insurance amount: "
    f"{len(non_self_pay_without_insurance):,}"
)

## 6. Cleaning and feature preparation

The cleaning stage keeps the relational structure intact and prepares the seven tables for SQL Server and Power BI.

Key transformations from the original workflow:

1. Standardize date columns to pandas datetime.
2. Recalculate `length_of_stay` from admission and discharge dates.
3. Correct patient registration dates when a later source event indicates an earlier valid date.
4. Correct treatment dates that fall after discharge by capping them at the discharge date.
5. Remove temporary data-quality helper columns before export.
6. Preserve the original table-level relationships and primary/foreign-key columns.

In [ ]:
# Clean patients
patients_clean = patients.copy()

first_appointment = (
    appointments.groupby("patient_id", as_index=False)["appointment_date"]
    .min()
    .rename(columns={"appointment_date": "first_appointment_date"})
)

first_admission = (
    admissions.groupby("patient_id", as_index=False)["admission_date"]
    .min()
    .rename(columns={"admission_date": "first_admission_date"})
)

patients_clean = (
    patients_clean
    .merge(first_appointment, on="patient_id", how="left")
    .merge(first_admission, on="patient_id", how="left")
)

patients_clean["registration_date"] = patients_clean[
    ["registration_date", "first_appointment_date", "first_admission_date"]
].min(axis=1)

patients_clean = patients_clean[
    [
        "patient_id",
        "age",
        "gender",
        "city",
        "insurance_type",
        "registration_date",
    ]
].copy()

In [ ]:
# Clean appointments
appointments_clean = appointments.copy()

# Remove temporary helper columns if they exist.
appointments_clean.drop(
    columns=["data_quality_issue"],
    errors="ignore",
    inplace=True,
)

# Ensure date/time fields have stable types.
appointments_clean["appointment_date"] = pd.to_datetime(
    appointments_clean["appointment_date"],
    errors="coerce",
)

In [ ]:
# Clean admissions
admissions_clean = admissions.copy()

admissions_clean["admission_date"] = pd.to_datetime(
    admissions_clean["admission_date"],
    errors="coerce",
)

admissions_clean["discharge_date"] = pd.to_datetime(
    admissions_clean["discharge_date"],
    errors="coerce",
)

admissions_clean["length_of_stay"] = (
    admissions_clean["discharge_date"]
    - admissions_clean["admission_date"]
).dt.days

In [ ]:
# Clean treatments
treatments_clean = treatments.merge(
    admissions[["admission_id", "admission_date", "discharge_date"]],
    on="admission_id",
    how="left",
    suffixes=("", "_admission"),
)

treatments_clean["treatment_date"] = pd.to_datetime(
    treatments_clean["treatment_date"],
    errors="coerce",
)

# Keep treatment dates within the admission window when a discharge date exists.
after_discharge = (
    treatments_clean["discharge_date"].notna()
    & (treatments_clean["treatment_date"] > treatments_clean["discharge_date"])
)

treatments_clean.loc[after_discharge, "treatment_date"] = (
    treatments_clean.loc[after_discharge, "discharge_date"]
)

treatments_clean = treatments_clean[
    [
        "treatment_id",
        "admission_id",
        "diagnosis",
        "treatment_type",
        "treatment_date",
        "treatment_cost",
    ]
].copy()

In [ ]:
# Clean billing, doctors and beds
billing_clean = billing.copy()
billing_clean.drop(columns=["calculated_total"], errors="ignore", inplace=True)

doctors_clean = doctors.copy()

beds_clean = beds.copy()
beds_clean.drop(columns=["data_quality_issue"], errors="ignore", inplace=True)

## 7. Final validation of cleaned tables

In [ ]:
clean_tables = {
    "patients_clean.csv": patients_clean,
    "doctors_clean.csv": doctors_clean,
    "appointments_clean.csv": appointments_clean,
    "admissions_clean.csv": admissions_clean,
    "treatments_clean.csv": treatments_clean,
    "billing_clean.csv": billing_clean,
    "beds_clean.csv": beds_clean,
}

clean_summary = pd.DataFrame(
    [
        {
            "file": filename,
            "rows": df.shape[0],
            "columns": df.shape[1],
            "missing_values": int(df.isna().sum().sum()),
        }
        for filename, df in clean_tables.items()
    ]
)

clean_summary

In [ ]:
# Confirm that primary-key columns remain unique after cleaning.
final_key_check = []

for filename, df in clean_tables.items():
    table_name = filename.replace("_clean.csv", "")
    key = primary_keys[table_name]

    final_key_check.append(
        {
            "table": table_name,
            "primary_key": key,
            "duplicate_rows": int(df[key].duplicated().sum()),
        }
    )

pd.DataFrame(final_key_check)

## 8. Export cleaned CSV files

These are the files used as the cleaned data layer for the downstream SQL Server and Power BI work.

In [ ]:
for filename, df in clean_tables.items():
    df.to_csv(CLEAN_DIR / filename, index=False)

print(f"Exported {len(clean_tables)} cleaned CSV files to: {CLEAN_DIR.resolve()}")

## 9. Exploratory analysis

The Python stage also provides quick checks that help understand the data before building SQL analysis and the Power BI dashboard.

In [ ]:
# Patient demographics
gender_summary = (
    patients_clean["gender"]
    .value_counts()
    .rename_axis("gender")
    .reset_index(name="total_patients")
)

gender_summary["percentage"] = (
    gender_summary["total_patients"]
    / gender_summary["total_patients"].sum()
    * 100
).round(2)

gender_summary

In [ ]:
# Admissions by type
admission_type_summary = (
    admissions_clean["admission_type"]
    .value_counts()
    .rename_axis("admission_type")
    .reset_index(name="total_admissions")
)

admission_type_summary["percentage"] = (
    admission_type_summary["total_admissions"]
    / admission_type_summary["total_admissions"].sum()
    * 100
).round(2)

admission_type_summary

In [ ]:
# Monthly admissions
monthly_admissions = (
    admissions_clean.assign(
        admission_year=admissions_clean["admission_date"].dt.year,
        month_number=admissions_clean["admission_date"].dt.month,
        month_name=admissions_clean["admission_date"].dt.month_name(),
    )
    .groupby(
        ["admission_year", "month_number", "month_name"],
        as_index=False,
    )
    .agg(total_admissions=("admission_id", "count"))
    .sort_values(["admission_year", "month_number"])
)

monthly_admissions

In [ ]:
# Monthly revenue
billing_with_admission_date = billing_clean.merge(
    admissions_clean[["admission_id", "admission_date"]],
    on="admission_id",
    how="left",
)

monthly_revenue = (
    billing_with_admission_date.assign(
        admission_year=billing_with_admission_date["admission_date"].dt.year,
        month_number=billing_with_admission_date["admission_date"].dt.month,
        month_name=billing_with_admission_date["admission_date"].dt.month_name(),
    )
    .groupby(
        ["admission_year", "month_number", "month_name"],
        as_index=False,
    )
    .agg(total_revenue=("total_amount", "sum"))
    .sort_values(["admission_year", "month_number"])
)

monthly_revenue

In [ ]:
# Department-level performance
department_summary = (
    admissions_clean
    .merge(
        billing_clean[["admission_id", "total_amount"]],
        on="admission_id",
        how="left",
    )
    .groupby("department", as_index=False)
    .agg(
        total_admissions=("admission_id", "count"),
        total_revenue=("total_amount", "sum"),
        avg_length_of_stay=("length_of_stay", "mean"),
    )
)

department_summary["revenue_per_admission"] = (
    department_summary["total_revenue"]
    / department_summary["total_admissions"]
)

department_summary.sort_values(
    "total_revenue",
    ascending=False,
)

In [ ]:
# Overall project KPIs
total_admissions = admissions_clean["admission_id"].nunique()
unique_patients = admissions_clean["patient_id"].nunique()
total_revenue = billing_clean["total_amount"].sum()
avg_bill_amount = billing_clean["total_amount"].mean()
avg_length_of_stay = admissions_clean["length_of_stay"].mean()

recovered = (admissions_clean["discharge_status"] == "Recovered").sum()
deceased = (admissions_clean["discharge_status"] == "Deceased").sum()

recovery_rate = recovered / total_admissions * 100
mortality_rate = deceased / total_admissions * 100

kpis = pd.Series(
    {
        "Total Admissions": total_admissions,
        "Unique Patients": unique_patients,
        "Total Revenue": total_revenue,
        "Average Bill Amount": avg_bill_amount,
        "Average Length of Stay": avg_length_of_stay,
        "Recovery Rate %": recovery_rate,
        "Mortality Rate %": mortality_rate,
    },
    name="value",
)

kpis

## 10. Handoff to SQL Server & Power BI

The cleaned CSVs exported above are the Python data-preparation output.

**Next stage — SQL Server**
- Load the seven cleaned tables.
- Preserve primary/foreign-key relationships.
- Perform business-oriented SQL analysis using joins, aggregations, CTEs, window functions, and date logic.

**Final stage — Power BI**
- Connect Power BI to the SQL Server database.
- Build measures and visuals from the relational model.
- Use slicers for department, admission type, and admission date.
- Present operational, financial, demographic, and outcome KPIs.

### Project outcome

Python establishes data quality and prepares the analytical layer; SQL performs the deeper business analysis; Power BI communicates the results through an interactive dashboard.